# Demo 2 bis — Il pitch non è il picco più alto

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
f0_mancante = 200.0
t_mancante = np.arange(2 * SR) / SR
armoniche = np.arange(1, 7)
pesi = np.array([0.5, 0.9, 1.0, 0.6, 0.4, 0.3])
componenti = np.sin(2 * np.pi * armoniche[:, None] * f0_mancante * t_mancante)
con_fondamentale = pesi @ componenti
pesi_senza = pesi.copy(); pesi_senza[0] = 0
senza_fondamentale = pesi_senza @ componenti
riferimento_pitch = np.sin(2 * np.pi * f0_mancante * t_mancante)
segnali_pitch = [riferimento_pitch, con_fondamentale, senza_fondamentale]
segnali_pitch = [x * (0.12 / np.sqrt(np.mean(x**2))) for x in segnali_pitch]

fig = make_subplots(rows=1, cols=2, subplot_titles=("con fondamentale", "senza fondamentale"))
for j, x_ in enumerate(segnali_pitch[1:], start=1):
    f, amp = al.amplitude_spectrum(x_[:SR], SR, window="boxcar")
    visibili = (f <= 1400) & (amp > 1e-8)
    fig.add_trace(go.Bar(x=f[visibili], y=amp[visibili], marker_color="#1769C2",
                         showlegend=False), row=1, col=j)
    fig.add_vline(x=f0_mancante, line=dict(color="#D47426", dash="dash"), row=1, col=j)
    fig.update_xaxes(range=[0, 1400], row=1, col=j)
fig.add_annotation(x=0.5, y=1.14, xref="paper", yref="paper", showarrow=False,
                   text="la linea arancione è l'altezza di riferimento (200 Hz)", font=dict(size=12))
fig.update_layout(height=360, title="Il pitch non è il picco più alto")
fig.show()
for nome, x in zip(["riferimento: 200 Hz", "con fondamentale", "senza fondamentale"], segnali_pitch):
    print(nome); display(al.safe_audio(al.fade_edges(x, SR), SR, normalize=False))

# La componente a 200 Hz è davvero assente; quella massima è a 600 Hz.
f, amp = al.amplitude_spectrum(segnali_pitch[2][:SR], SR, window="boxcar")
assert amp[np.argmin(abs(f - 200))] < 1e-10
assert f[np.argmax(amp)] == 600
